# NB1 — Delta Lake Basics

**Mục tiêu:** Tạo Delta table, observe transaction log, demo schema enforcement.

Maps to slide §2 (Delta Lake) + deliverable bullet 1.

In [1]:
import sys
sys.path.append("/workspace/scripts")
from spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark("nb1_delta_basics")

## 1. Write a Delta table

In [2]:
data = [
    (1, "alice", 30, "Hanoi"),
    (2, "bob", 25, "HCMC"),
    (3, "charlie", 35, "Danang"),
]
df = spark.createDataFrame(data, ["id", "name", "age", "city"])
table_path = "s3a://lakehouse/users_delta"
df.write.format("delta").mode("overwrite").save(table_path)

## 2. Read it back + inspect transaction log

Open MinIO console (http://localhost:9001) → `lakehouse/users_delta/_delta_log/`.
You should see `00000000000000000000.json`.

In [3]:
spark.read.format("delta").load(table_path).show()
spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`").show(truncate=False)

+---+-------+---+------+----+
| id|   name|age|  city|tier|
+---+-------+---+------+----+
|  3|charlie| 35|Danang|NULL|
|  1|  alice| 30| Hanoi|NULL|
|  2|    bob| 25|  HCMC|NULL|
+---+-------+---+------+----+

+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-------------------

## 2b. Verify `_delta_log/` via Hadoop FS (Spark path)

If MinIO console is unavailable, this prints the JSON files directly.

In [4]:
def list_delta_log(path: str) -> None:
    hadoop_conf = spark._jsc.hadoopConfiguration()
    delta_log = spark._jvm.org.apache.hadoop.fs.Path(path + "/_delta_log")
    fs = delta_log.getFileSystem(hadoop_conf)
    if fs.exists(delta_log):
        statuses = fs.listStatus(delta_log)
        json_files = sorted(
            [s.getPath().getName() for s in statuses if s.getPath().getName().endswith(".json")]
        )
        print(f"_delta_log JSON files: {len(json_files)}")
        print("Sample:", json_files[:5])
        return json_files
    else:
        print("No _delta_log found at", delta_log)
        return []

json_logs = list_delta_log(table_path)
assert len(json_logs) > 0, "No _delta_log JSON files found for the Delta table."

_delta_log JSON files: 6
Sample: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json', '00000000000000000003.json', '00000000000000000004.json']


## 3. Schema enforcement — try to write a wrong schema

In [5]:
schema_blocked = False
try:
    bad = spark.createDataFrame([(4, "dan", "thirty", "Hue")], ["id", "name", "age", "city"])
    bad.write.format("delta").mode("append").save(table_path)
except Exception as e:
    schema_blocked = True
    print("BLOCKED by schema enforcement (expected):")
    print(type(e).__name__, str(e)[:200])

if not schema_blocked:
    raise AssertionError("Schema enforcement did NOT block the bad write.")

BLOCKED by schema enforcement (expected):
AnalysisException [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'age' and 'age'


## 4. Schema evolution (opt-in)

In [6]:
new_col = spark.createDataFrame(
    [(4, "dan", 28, "Hue", "premium")],
    ["id", "name", "age", "city", "tier"],
)
new_col.write.format("delta").mode("append").option("mergeSchema", "true").save(table_path)
evolved = spark.read.format("delta").load(table_path)
evolved.show()
assert "tier" in evolved.columns, "Schema evolution failed: `tier` column was not added."
assert evolved.where("tier = 'premium'").count() >= 1, "Expected at least one `tier='premium'` row."

+---+-------+---+------+-------+
| id|   name|age|  city|   tier|
+---+-------+---+------+-------+
|  4|    dan| 28|   Hue|premium|
|  3|charlie| 35|Danang|   NULL|
|  1|  alice| 30| Hanoi|   NULL|
|  2|    bob| 25|  HCMC|   NULL|
+---+-------+---+------+-------+



## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] mergeSchema added the `tier` column

In [7]:
spark.stop()